# Tests des fonctions de calcul de la saturation

In [1]:
import json
from datetime import date, timedelta

import pandas as pd
from pandas import NamedAgg

#from saturation_image_quali_prod import (
from utils import (
    filter_sessions_duration,
    to_sampled_state_grp,
    to_state_grp,
    to_state_poc,
)
from e2_e3_e6 import (
    e2,
    e3,
    e6,
    filter_statuses_sessions,
    get_sampled_state_poc,
    get_state_poc_for_chunk,
    get_chunked_state_poc,
    get_chunked_state_grp,
    get_chunked_state_pools,
)
SAMPLES: int = 288  # 5 min
SATURE_H: int = 45  # minimum duration (min) of saturation to have a saturated hour
MAX_SESSION_DURATION_HOURS = 24

ID_POC: str = "id_pdc_itinerance"
ID_STATION: str = "id_station_itinerance"
ID_POOL: str = "id_pool"
SATURATION_RATIO = 0.1
OVERLOAD_RATIO = 0.2
MIN_POWER = 75

#day = date(2026,7, 5)
#day = date(2026,7, 14)
day = date(2026,8, 1)
date_file = f"{day.year}{day.month:02d}{day.day:02d}"

data_quali = "../data/"

In [2]:
def read_statics(day: date, min_power: float) -> pd.DataFrame:
    """Read static data for pocs and stations."""
    date_statics = f"{day.day:02d}-{day.month:02d}-{day.year}"
    e5_str = pd.read_csv(f"../data_DMR_e2_e3/e5_{date_statics}.csv")["extras"][0]
    statics = pd.DataFrame(json.loads(e5_str))
    statics["unite"] = statics["id_pdc_itinerance"].str[:5]
    return statics[statics["puissance_nominale"] >= min_power]

def read_statics_pools(day:date, min_power: float) -> pd.DataFrame:
    """Read static data for stations and pools."""
    e1_statics = pd.read_csv("../source/tests_DMR/aires_pdc_2026-07-25.csv")[[ID_POOL, ID_STATION]].drop_duplicates()
    return e1_statics

In [3]:
e1_statics = read_statics_pools(day, MIN_POWER)
e1_statics

,id_pool,id_station_itinerance
0,A000001,FRHPCPNF080371TIERSTOTEM
13,A000002,FRTSLP5670
14,A000002,FRIOYP13531046
29,A000003,FRIOYP13530804
51,A000004,FRFASP11568703
...,...,...
7112,A001871,NaN
7113,A001872,NaN
7114,A001873,NaN
7115,A001874,NaN


## test qualicharge

In [4]:
from datetime import datetime
    
samples_per_day = SAMPLES
min_power = MIN_POWER
chunk_size = 200

min_duration = timedelta(minutes=24 * 60 / samples_per_day)
max_duration = timedelta(hours=MAX_SESSION_DURATION_HOURS)

statuses = pd.read_parquet(data_quali + "qualicharge-" + date_file + "/statuses/production.parquet", engine="pyarrow")
sessions_s3 = pd.read_parquet(data_quali + "qualicharge-" + date_file + "/sessions/production.parquet", engine="pyarrow")
sessions = filter_sessions_duration(
    sessions_s3, min_duration=min_duration, max_duration=max_duration
)
statics = read_statics(day, MIN_POWER)
statics = statics[statics["puissance_nominale"] >= min_power]

sessions_poc = (
    sessions.groupby(ID_POC)
    .agg(
        sessions_nb=NamedAgg("energy", "count"),
        energy_cum=NamedAgg("energy", "sum"),
    )
    .reset_index()
)

In [5]:
# e2 indicator
sampled_state_poc, state_poc = get_chunked_state_poc(
    statics, day, samples_per_day, chunk_size, sessions, statuses
)
indicators_e2 = e2(
    #environment,
    state_poc,
    sessions_poc,
    day,
    #create_artifact,
    #persist,
)

In [6]:
# e3 indicator
state_station = get_chunked_state_grp(
    statics,
    sampled_state_poc,
    chunk_size,
    ID_STATION,
    SAMPLES,
    SATURATION_RATIO,
    OVERLOAD_RATIO,
    add_full_use=True,
    add_latency=True,
)
sessions_stations = pd.merge(
    statics[[ID_POC, ID_STATION]], sessions_poc, on=ID_POC, how="left"
).fillna(0)
info_sessions_stations = (
    sessions_stations[[ID_STATION, "sessions_nb", "energy_cum"]]
    .groupby(ID_STATION)
    .sum()
    .reset_index()
)
indicators_e3 = e3(
    #environment,
    state_station,
    info_sessions_stations,
    day,
    #create_artifact,
    #persist,
)

In [8]:
# e6 indicator
#pools_statics = get_station_pool_for_day(day, environment)
pools_statics = read_statics_pools(day, MIN_POWER)
pools_stations = pools_statics[[ID_POOL, ID_STATION]].drop_duplicates()
pools_pocs = pools_statics.merge(statics, on=ID_STATION, how="left")[[ID_POOL, ID_POC]]

state_pool = get_chunked_state_pools(
    statics,
    pools_stations,
    state_station,
    sampled_state_poc,
    chunk_size,
    SAMPLES,
    SATURATION_RATIO,
    OVERLOAD_RATIO,
    add_full_use=True,
    add_latency=True,
)
sessions_pools = pd.merge(
    pools_pocs, sessions_poc, on=ID_POC, how="left"
).fillna(0)
info_sessions_pools = (
    sessions_pools[[ID_POOL, "sessions_nb", "energy_cum"]]
    .groupby(ID_POOL)
    .sum()
    .reset_index()
)
indicators_e6 = e6(
    #environment,
    state_pool,
    info_sessions_pools,
    day,
    #create_artifact,
    #persist,
)


In [10]:
state_pool

,id_pool,nb_pdc,hs,inactif,pu_cum,sature_cum,surcharge,actif,sature_max,pu_max,pu_len
0,A000001,11.0,0.0,660.0,0.0,0.0,0.0,780.0,0.0,0.0,0.0
1,A000003,16.0,0.0,385.0,25.0,10.0,30.0,1015.0,5.0,25.0,25.0
2,A000004,8.0,0.0,320.0,330.0,90.0,155.0,875.0,35.0,60.0,330.0
3,A000005,8.0,0.0,585.0,120.0,25.0,75.0,755.0,15.0,50.0,45.0
4,A000007,12.0,0.0,470.0,0.0,0.0,0.0,970.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
1870,A001871,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1871,A001872,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1872,A001873,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1873,A001874,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
